# Component 1 — SHAP Temporal Drift Analysis
### AI-Based Electricity Demand Intelligence System · R26-IT-010 · SLIIT 2026

**Pipeline:**
1. Exploratory Data Analysis
2. Outlier Removal (IQR on target variable)
3. Train / Test Split (chronological 80/20)
4. Hyperparameter Tuning with Optuna (50 trials)
5. Final Model Training and Evaluation (RMSE, MAE, R²)
6. Save trained model as xgb_model.pkl
7. SHAP Temporal Drift (per-year feature importance in kW)

In [15]:
import pandas as pd
import numpy as np
import shap
import xgboost as xgb
import optuna
import joblib
import json
import os
import warnings
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# XGBoost 3.x â€” suppress output globally via config instead of constructor param
xgb.set_config(verbosity=0)

print('All libraries loaded successfully')

All libraries loaded successfully


## Step 1 â€” Exploratory Data Analysis

In [16]:
import os

# Locate the dataset relative to this notebook file's actual directory
# Works regardless of where VS Code sets the working directory
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('1_shap_temporal_drift.ipynb'))
DATASET_PATH = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', 'load_forecasting_dataset_corrected.csv'))

# Fallback: if above path doesn't exist, try workspace root directly
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = os.path.normpath(os.path.join(os.getcwd(), 'load_forecasting_dataset_corrected.csv'))

print(f'Dataset path : {DATASET_PATH}')
print(f'File exists  : {os.path.exists(DATASET_PATH)}')

df = pd.read_csv(DATASET_PATH)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)
df['Year'] = df['Timestamp'].dt.year

print()
print('=' * 55)
print('DATASET OVERVIEW')
print('=' * 55)
print(f'Total rows    : {len(df):,}')
print(f'Total columns : {df.shape[1]}')
print(f'Date range    : {df["Timestamp"].min().date()} to {df["Timestamp"].max().date()}')
print()
print('Rows per year:')
print(df['Year'].value_counts().sort_index().to_string())

Dataset path : c:\Users\dinuk\Desktop\Thanujan_Project\load_forecasting_dataset_corrected.csv
File exists  : True

DATASET OVERVIEW
Total rows    : 189,888
Total columns : 16
Date range    : 2020-01-01 to 2025-05-31

Rows per year:
Year
2020    35136
2021    35040
2022    35040
2023    35040
2024    35136
2025    14496


In [17]:
target_col = 'Load Demand (kW)'

print('Target variable â€” Load Demand (kW):')
print(df[target_col].describe().round(4).to_string())
print()

missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print('Missing values: None')
else:
    print('Missing values found:')
    print(missing.to_string())
print()

feature_cols_raw = [
    'Temperature (Â°C)', 'Humidity (%)', 'Wind Speed (m/s)', 'Rainfall (mm)',
    'Solar Irradiance (W/mÂ²)', 'GDP (LKR)', 'Per Capita Energy Use (kWh)',
    'Electricity Price (LKR/kWh)', 'Day of Week', 'Hour of Day', 'Month', 'Public Event'
]
print('Feature statistics:')
print(df[feature_cols_raw].describe().round(4).to_string())

Target variable â€” Load Demand (kW):
count    189888.0000
mean       1500.1544
std         199.9267
min         606.8792
25%        1365.2234
50%        1500.2750
75%        1635.0713
max        2412.4229

Missing values: None

Feature statistics:
       Temperature (Â°C)  Humidity (%)  Wind Speed (m/s)  Rainfall (mm)  Solar Irradiance (W/mÂ²)    GDP (LKR)  Per Capita Energy Use (kWh)  Electricity Price (LKR/kWh)  Day of Week  Hour of Day        Month  Public Event
count       189888.0000   189888.0000       189888.0000    189888.0000              189888.0000  189888.0000                  189888.0000                  189888.0000  189888.0000  189888.0000  189888.0000   189888.0000
mean            28.0015       79.9984            1.9961         5.0142                 249.9383    1000.0838                     499.9100                      24.9985       3.0010      11.5000       6.2533        0.0497
std              1.9993        5.0030            1.0023         5.0166                  5

## Step 2 â€” Outlier Removal

IQR method applied on the **target variable** (Load Demand) only.
XGBoost is tree-based and robust to feature-level outliers, so features are left untouched.
Extreme target spikes distort the model's understanding of normal demand patterns.

In [18]:
feature_cols = [
    'Temperature (Â°C)',
    'Humidity (%)',
    'Wind Speed (m/s)',
    'Rainfall (mm)',
    'Solar Irradiance (W/mÂ²)',
    'GDP (LKR)',
    'Per Capita Energy Use (kWh)',
    'Electricity Price (LKR/kWh)',
    'Day of Week',
    'Hour of Day',
    'Month',
    'Public Event'
]

feature_names = [
    'Temperature', 'Humidity', 'Wind Speed', 'Rainfall', 'Solar Irradiance',
    'GDP', 'Per Capita Energy Use', 'Electricity Price',
    'Day of Week', 'Hour of Day', 'Month', 'Public Event'
]

Q1  = df[target_col].quantile(0.25)
Q3  = df[target_col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_clean = df[
    (df[target_col] >= lower_bound) &
    (df[target_col] <= upper_bound)
].copy().reset_index(drop=True)

removed = len(df) - len(df_clean)

print('IQR Outlier Removal on Load Demand (kW)')
print(f'  Q1              : {Q1:.4f} kW')
print(f'  Q3              : {Q3:.4f} kW')
print(f'  IQR             : {IQR:.4f} kW')
print(f'  Lower bound     : {lower_bound:.4f} kW')
print(f'  Upper bound     : {upper_bound:.4f} kW')
print(f'  Rows before     : {len(df):,}')
print(f'  Rows removed    : {removed:,}  ({removed/len(df)*100:.2f}%)')
print(f'  Rows after      : {len(df_clean):,}')
print()
print('Target statistics after removal:')
print(df_clean[target_col].describe().round(4).to_string())

IQR Outlier Removal on Load Demand (kW)
  Q1              : 1365.2234 kW
  Q3              : 1635.0713 kW
  IQR             : 269.8479 kW
  Lower bound     : 960.4515 kW
  Upper bound     : 2039.8431 kW
  Rows before     : 189,888
  Rows removed    : 1,281  (0.67%)
  Rows after      : 188,607

Target statistics after removal:
count    188607.0000
mean       1500.3301
std         194.3819
min         960.5885
25%        1366.4462
50%        1500.3383
75%        1634.1147
max        2039.7656


## Step 3 â€” Train / Test Split

Chronological split: first 80% as training, last 20% as test.
Correct for time-series â€” prevents future data from leaking into training.

In [19]:
split_idx = int(len(df_clean) * 0.80)

train_df = df_clean.iloc[:split_idx].copy()
test_df  = df_clean.iloc[split_idx:].copy()

X_train = train_df[feature_cols].copy()
X_train.columns = feature_names
y_train = train_df[target_col].reset_index(drop=True)

X_test  = test_df[feature_cols].copy()
X_test.columns = feature_names
y_test  = test_df[target_col].reset_index(drop=True)

print('Train / Test Split (Chronological)')
print(f'  Training rows  : {len(X_train):,}  ({len(X_train)/len(df_clean)*100:.1f}%)')
print(f'  Test rows      : {len(X_test):,}   ({len(X_test)/len(df_clean)*100:.1f}%)')
print(f'  Train demand   : {y_train.min():.2f} to {y_train.max():.2f} kW')
print(f'  Test demand    : {y_test.min():.2f} to {y_test.max():.2f} kW')

Train / Test Split (Chronological)
  Training rows  : 150,885  (80.0%)
  Test rows      : 37,722   (20.0%)
  Train demand   : 960.59 to 2039.77 kW
  Test demand    : 961.06 to 2039.31 kW


## Step 4 â€” Hyperparameter Tuning with Optuna

Optuna runs Bayesian optimisation (TPE sampler) over the XGBoost hyperparameter space.
A holdout validation set (last 20% of training data) is used as the objective metric.
The test set remains completely untouched during this phase.

> 50 trials are used. This may take approximately 5 to 10 minutes.

In [20]:
val_idx = int(len(X_train) * 0.80)
X_tr    = X_train.iloc[:val_idx]
y_tr    = y_train.iloc[:val_idx]
X_val   = X_train.iloc[val_idx:]
y_val   = y_train.iloc[val_idx:]

# Use a subsample inside the Optuna objective for speed (10,000 rows is sufficient
# for Bayesian optimisation to find good hyperparameters â€” the final model trains on all data)
OPTUNA_SAMPLE = 10000
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_tr), size=min(OPTUNA_SAMPLE, len(X_tr)), replace=False)
X_tr_opt = X_tr.iloc[sample_idx]
y_tr_opt = y_tr.iloc[sample_idx]

print(f'Optuna internal split  Train: {len(X_tr):,}   Val: {len(X_val):,}')
print(f'Optuna training sample : {len(X_tr_opt):,} rows (subsampled for speed)')
print()

def objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators',      100, 500),
        'max_depth'       : trial.suggest_int('max_depth',           3,   8),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.30, log=True),
        'subsample'       : trial.suggest_float('subsample',        0.6,  1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6,  1.0),
        'min_child_weight': trial.suggest_int('min_child_weight',     1,  10),
        'reg_alpha'       : trial.suggest_float('reg_alpha',         0.0, 1.0),
        'reg_lambda'      : trial.suggest_float('reg_lambda',        0.0, 1.0),
        'random_state'    : 42,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_tr_opt, y_tr_opt)
    preds = model.predict(X_val)
    return float(np.sqrt(mean_squared_error(y_val, preds)))

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=50)

best_params = study.best_params.copy()
best_params['random_state'] = 42

print('Optuna Results')
print(f'  Trials completed     : {len(study.trials)}')
print(f'  Best validation RMSE : {study.best_value:.4f} kW')
print()
print('Best hyperparameters:')
for k, v in best_params.items():
    if k != 'random_state':
        print(f'  {k:<22}: {v}')

Optuna internal split  Train: 120,708   Val: 30,177
Optuna training sample : 10,000 rows (subsampled for speed)

Optuna Results
  Trials completed     : 50
  Best validation RMSE : 2.2463 kW

Best hyperparameters:
  n_estimators          : 386
  max_depth             : 3
  learning_rate         : 0.022146809028850203
  subsample             : 0.8400964659005544
  colsample_bytree      : 0.9584073809218324
  min_child_weight      : 6
  reg_alpha             : 0.20277594284005224
  reg_lambda            : 0.543569254368026


## Step 5 â€” Final Model Training

Trained on the **full training set** (not just the Optuna sub-split)
using the best hyperparameters found above.

In [21]:
final_model = xgb.XGBRegressor(**best_params)
final_model.fit(X_train, y_train)

print('Final model trained on full training set')
print(f'  Samples    : {len(X_train):,}')
print(f'  Features   : {len(feature_names)}')
print(f'  Estimators : {best_params["n_estimators"]}')
print(f'  Max depth  : {best_params["max_depth"]}')

Final model trained on full training set
  Samples    : 150,885
  Features   : 12
  Estimators : 386
  Max depth  : 3


## Step 6 â€” Model Evaluation on Test Set

Test set was never seen during training or Optuna tuning.
Train metrics are shown alongside test metrics to check for overfitting.

In [22]:
y_pred_test  = final_model.predict(X_test)
y_pred_train = final_model.predict(X_train)

test_rmse  = np.sqrt(mean_squared_error(y_test,  y_pred_test))
test_mae   = mean_absolute_error(y_test,  y_pred_test)
test_r2    = r2_score(y_test,  y_pred_test)

train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
train_mae  = mean_absolute_error(y_train, y_pred_train)
train_r2   = r2_score(y_train, y_pred_train)

print('=' * 55)
print('MODEL PERFORMANCE')
print('=' * 55)
print(f'{"Metric":<22} {"Train":>10} {"Test":>10}')
print('-' * 45)
print(f'{"RMSE (kW)":<22} {train_rmse:>10.4f} {test_rmse:>10.4f}')
print(f'{"MAE  (kW)":<22} {train_mae:>10.4f} {test_mae:>10.4f}')
print(f'{"R2 Score":<22} {train_r2:>10.4f} {test_r2:>10.4f}')
print('=' * 55)
print(f'Mean actual demand (test)  : {y_test.mean():.2f} kW')
print(f'RMSE as pct of mean demand : {test_rmse/y_test.mean()*100:.2f}%')
print(f'Train-Test RMSE gap        : {abs(train_rmse-test_rmse):.4f} kW')

MODEL PERFORMANCE
Metric                      Train       Test
---------------------------------------------
RMSE (kW)                  2.0442     2.0254
MAE  (kW)                  1.0552     1.0513
R2 Score                   0.9999     0.9999
Mean actual demand (test)  : 1499.65 kW
RMSE as pct of mean demand : 0.14%
Train-Test RMSE gap        : 0.0188 kW


## Step 6 — Save Trained Model

Saved as `.pkl` for reuse by other XAI components.

In [ ]:
import os

# Use absolute path so output always goes to the correct folder
# regardless of which directory VS Code uses as working directory
NOTEBOOK_FILE = os.path.abspath('1_shap_temporal_drift.ipynb')
NOTEBOOK_DIR  = os.path.dirname(NOTEBOOK_FILE)
output_dir    = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'backend', 'outputs'))
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(output_dir, 'xgb_model.pkl')
joblib.dump(final_model, model_path)

print(f'Output dir  : {output_dir}')
print(f'Model saved : {model_path}')
print(f'File size   : {os.path.getsize(model_path)/1024:.1f} KB')

## Step 8 â€” SHAP Temporal Drift Analysis

A separate XGBoost model is trained per year using the **Optuna best hyperparameters**.
SHAP TreeExplainer computes feature contributions in kW for each year's sample.
Mean absolute SHAP per feature reveals how importance shifts year by year.

In [24]:
SAMPLE_SIZE  = 2000
RANDOM_STATE = 42
years        = sorted(df_clean['Year'].unique())

temporal_results = {}

print('Computing SHAP values per year...')
print()

for year in years:
    year_df = df_clean[df_clean['Year'] == year][feature_cols + [target_col]].dropna()
    sample  = year_df.sample(min(SAMPLE_SIZE, len(year_df)), random_state=RANDOM_STATE)

    X_yr = sample[feature_cols].copy()
    X_yr.columns = feature_names
    y_yr = sample[target_col]

    # Train per-year model using the Optuna best hyperparameters (no re-tuning needed)
    yr_model = xgb.XGBRegressor(**best_params)
    yr_model.fit(X_yr, y_yr)

    explainer   = shap.TreeExplainer(yr_model)
    shap_values = explainer.shap_values(X_yr)
    mean_abs    = np.abs(shap_values).mean(axis=0)

    year_result = {
        feature_names[i]: round(float(mean_abs[i]), 4)
        for i in range(len(feature_names))
    }
    temporal_results[str(year)] = year_result

    top3 = sorted(year_result.items(), key=lambda x: x[1], reverse=True)[:3]
    print(f'  {year}  Top 3: {[(f, round(v,3)) for f, v in top3]}')

print()
print('All years processed')

Computing SHAP values per year...

  2020  Top 3: [('Temperature', 159.939), ('Wind Speed', 0.282), ('Hour of Day', 0.168)]
  2021  Top 3: [('Temperature', 157.928), ('Electricity Price', 0.239), ('Month', 0.157)]
  2022  Top 3: [('Temperature', 160.488), ('Wind Speed', 0.222), ('GDP', 0.117)]
  2023  Top 3: [('Temperature', 153.629), ('Wind Speed', 0.327), ('Solar Irradiance', 0.145)]
  2024  Top 3: [('Temperature', 157.171), ('GDP', 0.211), ('Hour of Day', 0.169)]
  2025  Top 3: [('Temperature', 157.27), ('Solar Irradiance', 0.142), ('Rainfall', 0.114)]

All years processed


## Step 9 â€” Compute Drift Insights

In [25]:
year_keys    = sorted(temporal_results.keys())
all_features = list(temporal_results[year_keys[0]].keys())

rankings = {}
for year in year_keys:
    sorted_feats = sorted(temporal_results[year].items(), key=lambda x: x[1], reverse=True)
    rankings[year] = [
        {'feature': f, 'shap': v, 'rank': i + 1}
        for i, (f, v) in enumerate(sorted_feats)
    ]

first_year = year_keys[0]
last_year  = year_keys[-1]
drift = {
    feat: round(temporal_results[last_year].get(feat, 0) - temporal_results[first_year].get(feat, 0), 4)
    for feat in all_features
}

biggest_gainer = max(drift.items(), key=lambda x: x[1])
biggest_loser  = min(drift.items(), key=lambda x: x[1])

avg_shap = {
    feat: round(np.mean([temporal_results[y].get(feat, 0) for y in year_keys]), 4)
    for feat in all_features
}
most_dominant = max(avg_shap.items(), key=lambda x: x[1])

print(f'Most dominant  : {most_dominant[0]}  avg SHAP {most_dominant[1]} kW')
print(f'Biggest gainer : {biggest_gainer[0]}  +{biggest_gainer[1]} kW  ({first_year} to {last_year})')
print(f'Biggest loser  : {biggest_loser[0]}  {biggest_loser[1]} kW  ({first_year} to {last_year})')

Most dominant  : Temperature  avg SHAP 157.7372 kW
Biggest gainer : Solar Irradiance  +0.1258 kW  (2020 to 2025)
Biggest loser  : Temperature  -2.6689 kW  (2020 to 2025)


## Step 10 â€” Save JSON Output

In [ ]:
output = {
    'years'           : year_keys,
    'features'        : all_features,
    'shap_by_year'    : temporal_results,
    'rankings_by_year': rankings,
    'drift'           : drift,
    'model_performance': {
        'train_rmse'   : round(train_rmse,        4),
        'test_rmse'    : round(test_rmse,          4),
        'train_mae'    : round(train_mae,          4),
        'test_mae'     : round(test_mae,           4),
        'train_r2'     : round(train_r2,           4),
        'test_r2'      : round(test_r2,            4),
        'train_size'   : len(X_train),
        'test_size'    : len(X_test),
        'optuna_trials': len(study.trials),
        'best_val_rmse': round(study.best_value,   4)
    },
    'insights': {
        'biggest_gainer': {
            'feature'  : biggest_gainer[0],
            'change'   : biggest_gainer[1],
            'from_year': first_year,
            'to_year'  : last_year
        },
        'biggest_loser': {
            'feature'  : biggest_loser[0],
            'change'   : biggest_loser[1],
            'from_year': first_year,
            'to_year'  : last_year
        },
        'most_dominant': {
            'feature' : most_dominant[0],
            'avg_shap': most_dominant[1]
        }
    }
}

json_path = os.path.join(output_dir, 'temporal_drift.json')
with open(json_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Saved : {json_path}')
print(f'Saved : {model_path}')

In [27]:
print('=' * 55)
print('COMPONENT 1 â€” COMPLETE SUMMARY')
print('=' * 55)
print()
print('Dataset')
print(f'  Original rows      : {len(df):,}')
print(f'  After outlier rem. : {len(df_clean):,}  (removed {len(df)-len(df_clean):,})')
print()
print('Split (Chronological)')
print(f'  Training set       : {len(X_train):,} rows')
print(f'  Test set           : {len(X_test):,} rows')
print()
print('Optuna Tuning')
print(f'  Trials             : {len(study.trials)}')
print(f'  Best val RMSE      : {study.best_value:.4f} kW')
print()
print('Model Performance â€” Test Set')
print(f'  RMSE               : {test_rmse:.4f} kW')
print(f'  MAE                : {test_mae:.4f} kW')
print(f'  R2                 : {test_r2:.4f}')
print(f'  RMSE / mean demand : {test_rmse/y_test.mean()*100:.2f}%')
print()
print('SHAP Temporal Drift')
print(f'  Years analysed     : {len(years)}  ({first_year} to {last_year})')
print(f'  Most dominant      : {most_dominant[0]}  avg {most_dominant[1]} kW')
print(f'  Biggest gainer     : {biggest_gainer[0]}  +{biggest_gainer[1]} kW')
print(f'  Biggest loser      : {biggest_loser[0]}  {biggest_loser[1]} kW')
print()
print('Output Files')
print('  ../backend/outputs/temporal_drift.json')
print('  ../backend/outputs/xgb_model.pkl')

COMPONENT 1 â€” COMPLETE SUMMARY

Dataset
  Original rows      : 189,888
  After outlier rem. : 188,607  (removed 1,281)

Split (Chronological)
  Training set       : 150,885 rows
  Test set           : 37,722 rows

Optuna Tuning
  Trials             : 50
  Best val RMSE      : 2.2463 kW

Model Performance â€” Test Set
  RMSE               : 2.0254 kW
  MAE                : 1.0513 kW
  R2                 : 0.9999
  RMSE / mean demand : 0.14%

SHAP Temporal Drift
  Years analysed     : 6  (2020 to 2025)
  Most dominant      : Temperature  avg 157.7372 kW
  Biggest gainer     : Solar Irradiance  +0.1258 kW
  Biggest loser      : Temperature  -2.6689 kW

Output Files
  ../backend/outputs/temporal_drift.json
  ../backend/outputs/xgb_model.pkl
